# Model Selection & Multi-Architecture Benchmarking

Trains and compares **CatBoost, XGBoost, LightGBM, and Random Forest** on 60,000 records with grouped leakage-free validation.


In [ ]:
import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
from feature_engineering import PanchayatFeatureEngineer
from preprocessing import PanchayatPreprocessor, split_dataset

# 1. Load Data
df = pd.read_parquet('../datasets/synthetic/synthetic_panchayats_all_5_states.parquet')
fe = PanchayatFeatureEngineer()
df_fe = fe.transform(df)

# 2. Grouped Split (Zero Leakage)
train_df, test_df = split_dataset(df_fe, test_size=0.20, seed=42)

# 3. Preprocessing
prep = PanchayatPreprocessor()
prep.fit(train_df)
X_train = prep.transform_features(train_df)
X_test = prep.transform_features(test_df)

y_train = prep.target_encoders['priority_sector'].transform(train_df['priority_sector'])
y_test = prep.target_encoders['priority_sector'].transform(test_df['priority_sector'])


## 1. Train and Benchmark Classifiers


In [ ]:
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
import catboost as cb
from sklearn.metrics import accuracy_score, classification_report

models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    'LightGBM': lgb.LGBMClassifier(n_estimators=150, learning_rate=0.08, random_state=42, verbose=-1),
    'XGBoost': xgb.XGBClassifier(n_estimators=150, learning_rate=0.08, random_state=42, eval_metric='mlogloss', n_jobs=-1),
    'CatBoost': cb.CatBoostClassifier(iterations=180, learning_rate=0.08, random_seed=42, verbose=0),
}

for name, m in models.items():
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    if hasattr(preds, 'flatten'):
        preds = preds.flatten().astype(int)
    acc = accuracy_score(y_test, preds)
    print(f'{name} Test Accuracy: {acc*100:.2f}%')


## 2. Train Clustering and Anomaly Detection


In [ ]:
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest

kmeans = KMeans(n_clusters=4, random_state=42).fit(X_train)
iso = IsolationForest(contamination=0.03, random_state=42).fit(X_train)

print('KMeans Cluster Centers Shape:', kmeans.cluster_centers_.shape)
print('Anomalies in Test Set:', (iso.predict(X_test) == -1).sum())
